In [ ]:
import pandas as pd
import datetime as dt
import os

file_path = os.path.join(os.path.expanduser("~"), "Downloads", "ınvoices_clean.csv")
output_path = os.path.join(os.path.expanduser("~"), "Downloads", "RFMskorlamaVeEtiketleme.csv")

try:
    df = pd.read_csv(file_path, sep=',')
    df.columns = df.columns.str.strip()

    df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
    today_date = df['InvoiceDate'].max() + dt.timedelta(days=2)

    #Recency, Frequency ve Monetary Hesaplama
    rfm = df.groupby('Customer_id').agg({
        'InvoiceDate': lambda date: (today_date - date.max()).days,
        'Invoice': lambda num: num.nunique(),
        'Total_Price': lambda price: price.sum()
    })
    
    rfm.columns = ['recency', 'frequency', 'monetary']

    rfm['monetary'] = rfm['monetary'].round(2).astype(float)

    #1-5 Arası Puanlama
    rfm["recency_score"] = pd.qcut(rfm['recency'], 5, labels=[5, 4, 3, 2, 1])
    rfm["frequency_score"] = pd.qcut(rfm['frequency'].rank(method="first"), 5, labels=[1, 2, 3, 4, 5])
    rfm["monetary_score"] = pd.qcut(rfm['monetary'], 5, labels=[1, 2, 3, 4, 5])

    #RF Skorunu Oluşturma
    rfm["RF_SCORE"] = (rfm['recency_score'].astype(str) + rfm['frequency_score'].astype(str))

    #Segment Etiketlerinin Atanması
    seg_map = {
        r'[1-2][1-2]': 'hibernating',
        r'[1-2][3-4]': 'at_Risk',
        r'[1-2]5': 'cant_loose',
        r'3[1-2]': 'about_to_sleep',
        r'33': 'need_attention',
        r'[3-4][4-5]': 'loyal_customers',
        r'41': 'promising',
        r'51': 'new_customers',
        r'[4-5][2-3]': 'potential_loyalists',
        r'5[4-5]': 'champions'
    }

    rfm['segment'] = rfm['RF_SCORE'].replace(seg_map, regex=True)

    print("\nRFM Skorlama & Etiketleme")
    print(rfm['segment'].value_counts())

    rfm.to_csv(output_path, sep=';', encoding='utf-8-sig')

except KeyError as e:
    print(f"Hata: Veri seti içerisinde beklenen kolon bulunamadı -> {e}")
except Exception as e:
    print(f"Beklenmeyen bir hata oluştu: {e}")


--- RFM Skorlama & Etiketleme Tamamlandı ---
segment
hibernating            1518
loyal_customers        1156
champions               837
at_Risk                 755
potential_loyalists     714
about_to_sleep          384
need_attention          268
promising               115
cant_loose               72
new_customers            54
Name: count, dtype: int64


In [4]:
print(f"Toplam İşlem Sayısı: {len(df)}")
print(f"Benzersiz Müşteri Sayısı: {df['Customer_id'].nunique()}")

Toplam İşlem Sayısı: 779325
Benzersiz Müşteri Sayısı: 5873
